In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Dataset_English_Hindi.csv")
df = df.dropna()
df = df.drop_duplicates()
df = df[df['English'].str.len() < 300]
df = df[df['Hindi'].str.len() < 300]
df.to_csv("/content/drive/MyDrive/cleaned_dataset_en_hi.csv", index=False)

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)  # keep only letters/numbers
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['English'] = df['English'].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(df, test_size=0.2, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)

train.to_csv("/content/drive/MyDrive/train.csv", index=False)
val.to_csv("/content/drive/MyDrive/val.csv", index=False)
test.to_csv("/content/drive/MyDrive/test.csv", index=False)


In [ ]:
import json

data = []
for _, row in train.iterrows():
    data.append({"translation": {"en": row["English"], "hi": row["Hindi"]}})

with open("train.json", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")


In [ ]:
data

[{'translation': {'en': 'it is advantageous in cases involving disputed questions of facts for example whether goods are up to the original samples or in the assessment of damages or whether compensation is reasonable',
   'hi': 'ऐसे मामलों में जिनमें तथ्यों के प्रश्नों पर विवाद है , जैसे कि माल मूल नमूनों के अनुरूप है या नहीं , नुकसानी का निर्धारण कैसे किया जाए , या प्रतिकर की राशि उचित है या नहीं , माध्यस्थम् की शरण लेना लाभकारी रहता है .'}},
 {'translation': {'en': 'in india communism and socialism are understood by relatively very few persons and most people who shout loudest against them are supremely ignorant about them',
   'hi': 'हिंदुस्तान में साम्यवाद और समाजवाद को आमतौर पर बहुत कम लोग समझते हैं.वे तो इनके बारे में कुछ भी नहीं जानते हैं .'}},
 {'translation': {'en': 'along with these there are many small industries also which give employment to many people in small indian villages and cities',
   'hi': 'इसके साथ ही कई लघु स्तर के उद्योग भी हैं जोकि छोटे भारतीय गाँव और भारतीय 

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

# Load model and tokenizer for English to Hindi translation
model_name_en_hi = "Helsinki-NLP/opus-mt-en-hi"
tokenizer_en_hi = MarianTokenizer.from_pretrained(model_name_en_hi)
model_en_hi = MarianMTModel.from_pretrained(model_name_en_hi)

def translate_en_to_hi(text_to_translate):
    tokens = tokenizer_en_hi([text_to_translate], return_tensors="pt", padding=True, truncation=True, max_length=512)
    translated_ids = model_en_hi.generate(**tokens)
    decoded_hindi = tokenizer_en_hi.decode(translated_ids[0], skip_special_tokens=True)
    return decoded_hindi

# Example usage for English to Hindi
english_input = input("Enter your sentence: ")
hindi_output = translate_en_to_hi(english_input)
print(f"English Input: {english_input}")
print(f"Hindi Output: {hindi_output}")


# Load model and tokenizer for Hindi to English translation
model_name_hi_en = "Helsinki-NLP/opus-mt-hi-en"
tokenizer_hi_en = MarianTokenizer.from_pretrained(model_name_hi_en)
model_hi_en = MarianMTModel.from_pretrained(model_name_hi_en)

def translate_hi_to_en(text_to_translate):
    tokens = tokenizer_hi_en([text_to_translate], return_tensors="pt", padding=True, truncation=True, max_length=512)
    translated_ids = model_hi_en.generate(**tokens)
    decoded_english = tokenizer_hi_en.decode(translated_ids[0], skip_special_tokens=True)
    return decoded_english

# Example usage for Hindi to English
hindi_input = input("Enter your sentence: ")
english_output_from_hi = translate_hi_to_en(hindi_input)
print(f"Hindi Input: {hindi_input}")
print(f"English Output: {english_output_from_hi}")

### Hindi Sample Sentence: नमस्ते, आप आज कैसे हैं?

In [ ]:
# import torch
# import torch.nn as nn
# from torch.utils.data import Dataset, DataLoader
# import pandas as pd
# import nltk
# from collections import Counter
# import tkinter as tk
# from tkinter import ttk
#
# # Load data
# df = pd.read_csv('multilingual_data.csv')
# languages = ['English', 'Hindi', 'Marathi', 'Tamil', 'Chinese', 'Japanese', 'Korean', 'Malayalam', 'Telugu', 'German']
# lang_to_id = {lang: i for i, lang in enumerate(languages)}
#
#
# # Tokenize function with error handling
# def tokenize(text):
#     try:
#         return nltk.word_tokenize(text.lower())
#     except LookupError:
#         return text.lower().split()
#
#
# df['source_tokens'] = df['source_text'].apply(tokenize)
# df['target_tokens'] = df['target_text'].apply(tokenize)
#
# # Build shared vocab
# all_tokens = df['source_tokens'].tolist() + df['target_tokens'].tolist()
# counter = Counter(word for sent in all_tokens for word in sent)
# vocab = {word: idx + 4 for idx, (word, freq) in enumerate(counter.items()) if freq >= 2}
# vocab.update({'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3})
# inv_vocab = {v: k for k, v in vocab.items()}
#
#
# # Dataset
# class MultiLangDataset(Dataset):
#     def __init__(self, df, vocab):
#         self.df = df
#         self.vocab = vocab
#
#     def __len__(self):
#         return len(self.df)
#
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         src = [self.vocab.get(w, 3) for w in row['source_tokens'][:50]]
#         tgt = [1] + [self.vocab.get(w, 3) for w in row['target_tokens'][:50]] + [2]
#         src_lang = lang_to_id[row['source_lang']]
#         tgt_lang = lang_to_id[row['target_lang']]
#         return torch.tensor(src), torch.tensor(tgt), src_lang, tgt_lang
#
#
# # Fixed collate_fn
# def collate_fn(batch):
#     src_batch = [item[0] for item in batch]
#     tgt_batch = [item[1] for item in batch]
#     src_lang_batch = [item[2] for item in batch]
#     tgt_lang_batch = [item[3] for item in batch]
#
#     src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
#     tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
#
#     src_lang_tensor = torch.tensor(src_lang_batch)
#     tgt_lang_tensor = torch.tensor(tgt_lang_batch)
#
#     return src_padded, tgt_padded, src_lang_tensor, tgt_lang_tensor
#
#
# dataset = MultiLangDataset(df, vocab)
# dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
#
#
# # Fixed Encoder: Use concatenation for embeddings
# class Encoder(nn.Module):
#     def __init__(self, vocab_size, hidden_size, num_langs):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, hidden_size)
#         self.lang_emb = nn.Embedding(num_langs, hidden_size)
#         self.rnn = nn.GRU(hidden_size * 2, hidden_size, batch_first=True)  # input_size = hidden_size * 2
#
#     def forward(self, x, lang):
#         word_emb = self.embedding(x)  # (batch, seq, hidden_size)
#         lang_emb = self.lang_emb(lang).unsqueeze(1).expand(-1, x.size(1), -1)  # (batch, seq, hidden_size)
#         emb = torch.cat([word_emb, lang_emb], dim=-1)  # (batch, seq, hidden_size * 2)
#         _, hidden = self.rnn(emb)
#         return hidden
#
#
# # Fixed Decoder: Use concatenation for embeddings
# class Decoder(nn.Module):
#     def __init__(self, vocab_size, hidden_size, num_langs):
#         super().__init__()
#         self.embedding = nn.Embedding(vocab_size, hidden_size)
#         self.lang_emb = nn.Embedding(num_langs, hidden_size)
#         self.rnn = nn.GRU(hidden_size * 2, hidden_size, batch_first=True)  # input_size = hidden_size * 2
#         self.fc = nn.Linear(hidden_size, vocab_size)
#
#     def forward(self, x, hidden, lang):
#         word_emb = self.embedding(x.unsqueeze(1))  # (batch, 1, hidden_size)
#         lang_emb = self.lang_emb(lang).unsqueeze(1)  # (batch, 1, hidden_size)
#         emb = torch.cat([word_emb, lang_emb], dim=-1)  # (batch, 1, hidden_size * 2)
#         output, hidden = self.rnn(emb, hidden)
#         return self.fc(output.squeeze(1)), hidden
#
#
# class Seq2Seq(nn.Module):
#     def __init__(self, encoder, decoder):
#         super().__init__()
#         self.encoder = encoder
#         self.decoder = decoder
#
#     def forward(self, src, tgt, src_lang, tgt_lang):
#         hidden = self.encoder(src, src_lang)
#         outputs = []
#         input = tgt[:, 0]
#         for t in range(1, tgt.size(1)):
#             output, hidden = self.decoder(input, hidden, tgt_lang)
#             outputs.append(output.unsqueeze(1))
#             input = tgt[:, t]
#         return torch.cat(outputs, dim=1)
#
#
# # Train
# model = Seq2Seq(Encoder(len(vocab), 256, len(languages)), Decoder(len(vocab), 256, len(languages)))
# optimizer = torch.optim.Adam(model.parameters())
# criterion = nn.CrossEntropyLoss(ignore_index=0)
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model.to(device)
#
# for epoch in range(5):
#     for src, tgt, src_lang, tgt_lang in dataloader:
#         src, tgt = src.to(device), tgt.to(device)
#         src_lang, tgt_lang = src_lang.to(device), tgt_lang.to(device)
#         optimizer.zero_grad()
#         output = model(src, tgt, src_lang, tgt_lang)
#         loss = criterion(output.view(-1, len(vocab)), tgt[:, 1:].contiguous().view(-1))
#         loss.backward()
#         optimizer.step()
#     print(f'Epoch {epoch + 1} Loss: {loss.item()}')
#
# torch.save(model.state_dict(), 'multilingual_seq2seq.pth')
#
#
# # Translation function
# def translate(src_lang, tgt_lang, text):
#     model.eval()
#     tokens = [vocab.get(w, 3) for w in tokenize(text)][:50]
#     src = torch.tensor(tokens).unsqueeze(0).to(device)
#     src_id = torch.tensor([lang_to_id[src_lang]]).to(device)
#     tgt_id = torch.tensor([lang_to_id[tgt_lang]]).to(device)
#     hidden = model.encoder(src, src_id)
#     input = torch.tensor([1]).to(device)
#     outputs = []
#     for _ in range(50):
#         output, hidden = model.decoder(input, hidden, tgt_id)
#         top1 = output.argmax(1)
#         if top1.item() == 2:
#             break
#         outputs.append(inv_vocab[top1.item()])
#         input = top1
#     return ' '.join(outputs)
#
#
# # Tkinter UI
# root = tk.Tk()
# root.title("Multi-Language Translator")
# root.geometry("400x300")
#
# tk.Label(root, text="Source Language:").pack()
# src_var = tk.StringVar()
# src_dropdown = ttk.Combobox(root, textvariable=src_var, values=languages)
# src_dropdown.pack()
#
# tk.Label(root, text="Target Language:").pack()
# tgt_var = tk.StringVar()
# tgt_dropdown = ttk.Combobox(root, textvariable=tgt_var, values=languages)
# tgt_dropdown.pack()
#
# tk.Label(root, text="Input Text:").pack()
# input_text = tk.Text(root, height=3)
# input_text.pack()
#
#
# def on_translate():
#     src = src_var.get()
#     tgt = tgt_var.get()
#     text = input_text.get("1.0", tk.END).strip()
#     if src and tgt and text:
#         result = translate(src, tgt, text)
#         output_text.delete("1.0", tk.END)
#         output_text.insert(tk.END, result)
#
#
# tk.Button(root, text="Translate", command=on_translate).pack()
#
# tk.Label(root, text="Translated Text:").pack()
# output_text = tk.Text(root, height=3)
# output_text.pack()
#
# root.mainloop()


ModuleNotFoundError: No module named 'streamlit'

In [ ]:


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import nltk
from collections import Counter
import tkinter as tk
from tkinter import ttk
from sacrebleu import corpus_bleu

# Load data
df = pd.read_csv('multilingual_data.csv')
languages = ['English', 'Hindi', 'Marathi', 'Tamil', 'Telugu', 'Malayalam','Japanese', 'Korean', 'Chinese', 'German']
lang_to_id = {lang: i for i, lang in enumerate(languages)}


def tokenize(text):
    try:
        return nltk.word_tokenize(text.lower())
    except LookupError:
        return text.lower().split()


df['source_tokens'] = df['source_text'].apply(tokenize)
df['target_tokens'] = df['target_text'].apply(tokenize)

# Split train/val
train_df = df.sample(frac=0.8, random_state=42)
val_df = df.drop(train_df.index)

# Build vocab (FIXED: Filter counter first)
all_tokens = df['source_tokens'].tolist() + df['target_tokens'].tolist()
counter = Counter(word for sent in all_tokens for word in sent)
filtered_counter = {w: f for w, f in counter.items() if f >= 2}
vocab = {word: idx + 4 for idx, word in enumerate(filtered_counter)}
vocab.update({'<pad>': 0, '<sos>': 1, '<eos>': 2, '<unk>': 3})
inv_vocab = {v: k for k, v in vocab.items()}

# DEBUG: Check vocab and IDs
# print(f"Vocab size: {len(vocab)}")
# print(f"Sample vocab: {list(vocab.items())[:5]}")
# max_id = max(
#     [vocab.get(w, 3) for sent in df['source_tokens'] for w in sent] + [vocab.get(w, 3) for sent in df['target_tokens']
#                                                                        for w in sent])
# print(f"Max token ID in data: {max_id}")
# if max_id >= len(vocab):
#     print("ERROR: Token ID exceeds vocab size!")
#     exit()


# Dataset
class MultiLangDataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        src = [self.vocab.get(w, 3) for w in row['source_tokens'][:50]]
        tgt = [1] + [self.vocab.get(w, 3) for w in row['target_tokens'][:50]] + [2]
        src_lang = lang_to_id[row['source_lang']]
        tgt_lang = lang_to_id[row['target_lang']]
        return torch.tensor(src), torch.tensor(tgt), src_lang, tgt_lang


def collate_fn(batch):
    src_batch = [item[0] for item in batch]
    tgt_batch = [item[1] for item in batch]
    src_lang_batch = [item[2] for item in batch]
    tgt_lang_batch = [item[3] for item in batch]
    src_padded = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_padded = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    src_lang_tensor = torch.tensor(src_lang_batch)
    tgt_lang_tensor = torch.tensor(tgt_lang_batch)
    return src_padded, tgt_padded, src_lang_tensor, tgt_lang_tensor


train_dataset = MultiLangDataset(train_df, vocab)
val_dataset = MultiLangDataset(val_df, vocab)
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)


# Attention
class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        hidden = hidden.squeeze(0)
        seq_len = encoder_outputs.size(1)
        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy = torch.tanh(self.attn(torch.cat([hidden, encoder_outputs], dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)


# Encoder
class Encoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_langs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lang_emb = nn.Embedding(num_langs, hidden_size)
        self.rnn = nn.GRU(hidden_size * 2, hidden_size, batch_first=True)

    def forward(self, x, lang):
        word_emb = self.embedding(x)
        lang_emb = self.lang_emb(lang).unsqueeze(1).expand(-1, x.size(1), -1)
        emb = torch.cat([word_emb, lang_emb], dim=-1)
        outputs, hidden = self.rnn(emb)
        return outputs, hidden


# Decoder
class Decoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_langs):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lang_emb = nn.Embedding(num_langs, hidden_size)
        self.attention = Attention(hidden_size)
        self.rnn = nn.GRU(hidden_size * 3, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden, lang, encoder_outputs):
        word_emb = self.embedding(x.unsqueeze(1))
        lang_emb = self.lang_emb(lang).unsqueeze(1)
        attn_weights = self.attention(hidden, encoder_outputs)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        emb = torch.cat([word_emb, lang_emb, context], dim=-1)
        output, hidden = self.rnn(emb, hidden)
        return self.fc(output.squeeze(1)), hidden


# Seq2Seq
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, src_lang, tgt_lang):
        encoder_outputs, hidden = self.encoder(src, src_lang)
        outputs = []
        input = tgt[:, 0]
        for t in range(1, tgt.size(1)):
            output, hidden = self.decoder(input, hidden, tgt_lang, encoder_outputs)
            outputs.append(output.unsqueeze(1))
            input = tgt[:, t]
        return torch.cat(outputs, dim=1)


# Train
model = Seq2Seq(Encoder(len(vocab), 512, len(languages)), Decoder(len(vocab), 512, len(languages)))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for src, tgt, src_lang, tgt_lang in train_dataloader:
        src, tgt = src.to(device), tgt.to(device)
        src_lang, tgt_lang = src_lang.to(device), tgt_lang.to(device)
        optimizer.zero_grad()
        output = model(src, tgt, src_lang, tgt_lang)
        loss = criterion(output.view(-1, len(vocab)), tgt[:, 1:].contiguous().view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    # Validation
    model.eval()
    val_predictions = []
    val_references = []
    with torch.no_grad():
        for src, tgt, src_lang, tgt_lang in val_dataloader:
            src, tgt = src.to(device), tgt.to(device)
            src_lang, tgt_lang = src_lang.to(device), tgt_lang.to(device)
            output = model(src, tgt, src_lang, tgt_lang)
            for i in range(src.size(0)):
                pred = []
                hidden = model.encoder(src[i:i + 1], src_lang[i:i + 1])[1]
                input = torch.tensor([1]).to(device)
                for _ in range(50):
                    out, hidden = model.decoder(input, hidden, tgt_lang[i:i + 1],
                                                model.encoder(src[i:i + 1], src_lang[i:i + 1])[0])
                    top1 = out.argmax(1)
                    if top1.item() == 2:
                        break
                    pred.append(inv_vocab[top1.item()])
                    input = top1
                val_predictions.append(' '.join(pred))
                val_references.append(' '.join([inv_vocab[t.item()] for t in tgt[i, 1:] if t.item() not in [0, 2]]))
    bleu = corpus_bleu(val_predictions, [val_references])
    print(f'Epoch {epoch + 1}, Train Loss: {total_loss / len(train_dataloader):.4f}, BLEU: {bleu.score:.2f}')

torch.save(model.state_dict(), 'multilingual_seq2seq.pth')


# Translation
def translate(src_lang, tgt_lang, text):
    model.eval()
    tokens = [vocab.get(w, 3) for w in tokenize(text)][:50]
    src = torch.tensor(tokens).unsqueeze(0).to(device)
    src_id = torch.tensor([lang_to_id[src_lang]]).to(device)
    tgt_id = torch.tensor([lang_to_id[tgt_lang]]).to(device)
    encoder_outputs, hidden = model.encoder(src, src_id)
    input = torch.tensor([1]).to(device)
    outputs = []
    for _ in range(50):
        output, hidden = model.decoder(input, hidden, tgt_id, encoder_outputs)
        top1 = output.argmax(1)
        if top1.item() == 2:
            break
        outputs.append(inv_vocab[top1.item()])
        input = top1
    return ' '.join(outputs)


# Tkinter UI
root = tk.Tk()
root.title("Multi-Language Translator")
root.geometry("400x300")

tk.Label(root, text="Source Language:").pack()
src_var = tk.StringVar()
src_dropdown = ttk.Combobox(root, textvariable=src_var, values=languages)
src_dropdown.pack()

tk.Label(root, text="Target Language:").pack()
tgt_var = tk.StringVar()
tgt_dropdown = ttk.Combobox(root, textvariable=tgt_var, values=languages)
tgt_dropdown.pack()

tk.Label(root, text="Input Text:").pack()
input_text = tk.Text(root, height=3)
input_text.pack()


def on_translate():
    src = src_var.get()
    tgt = tgt_var.get()
    text = input_text.get("1.0", tk.END).strip()
    if src and tgt and text:
        result = translate(src, tgt, text)
        output_text.delete("1.0", tk.END)
        output_text.insert(tk.END, result)


tk.Button(root, text="Translate", command=on_translate).pack()

tk.Label(root, text="Translated Text:").pack()
output_text = tk.Text(root, height=3)
output_text.pack()

root.mainloop()
